## Install SQL

In [61]:
! pip install pymysql

In [62]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [91]:
%reload_ext sql

In [137]:
%sql sqlite:///1SQL_Joins_Assignment.sql

In [154]:
import sqlite3
import re

# Create a proper SQLite database file and connect to it
conn = sqlite3.connect('joins_assignment.db')
cursor = conn.cursor()

# Read the raw SQL statements from your assignment file
with open('1SQL_Joins_Assignment.sql', 'r') as f:
    sql_script = f.read()

# Clean MySQL specific commands like 'create database' and 'use' so SQLite can parse it
sql_script_clean = re.sub(r'(?i)create\s+database\s+[^;]+;', '', sql_script)
sql_script_clean = re.sub(r'(?i)use\s+[^;]+;', '', sql_script_clean)

# Execute the script to create tables and insert data
try:
    cursor.executescript(sql_script_clean)
    conn.commit()
    print("Database successfully initialized with the SQL script!")
except Exception as e:
    print(f"An error occurred while executing the script: {e}")
finally:
    conn.close()

An error occurred while executing the script: table Customers already exists


In [143]:
%sql sqlite:///joins_assignment.db

Connecting and switching to connection 'sqlite:///joins_assignment.db'

Sample question to check tables?

In [148]:
%sql SELECT * FROM Customers limit 5;

Running query in 'sqlite:///joins_assignment.db'

CustomerID,CustomerName,City,JoinDate
101,Rahul Sharma,Bangalore,2023-01-15
102,Priya Mehta,Mumbai,2024-03-20
103,Arjun Nair,Chennai,2025-01-12
104,Sneha Gupta,Delhi,2022-07-10
105,Kavya Reddy,Hyderabad,2024-08-15


**Question 1**
A retail company wants to reward its most valuable customers. Display the
top 5 highest-value orders along with customer names.

In [176]:
%sql select c.CustomerID, c.CustomerName,o.OrderID, o.OrderAmount from Orders o join Customers c on o.CustomerID = c.CustomerID order by OrderAmount desc Limit 5;


Running query in 'sqlite:///joins_assignment.db'

CustomerID,CustomerName,OrderID,OrderAmount
110,Vikram Patel,5011,42000
101,Rahul Sharma,5009,30000
102,Priya Mehta,5014,27000
106,Amit Verma,5007,25000
101,Rahul Sharma,5003,22000


Question 2
The marketing team wants to send promotional emails to customers whose
names start with 'A', 'R', or 'S'.
Write a query without using multiple OR conditions

In [158]:
%sql select CustomerID,CustomerName,City,JoinDate from Customers where substr(CustomerName, 1, 1) in ('A','R','S');


Running query in 'sqlite:///joins_assignment.db'

CustomerID,CustomerName,City,JoinDate
101,Rahul Sharma,Bangalore,2023-01-15
103,Arjun Nair,Chennai,2025-01-12
104,Sneha Gupta,Delhi,2022-07-10
106,Amit Verma,Pune,2023-11-05
107,Rohan Singh,Kolkata,2024-05-18
108,Simran Kaur,Delhi,2025-02-01
109,Anjali Jain,Mumbai,2023-09-22


**Question 3**
Management wants to identify customers who have placed orders worth
more than ₹10,000 but whose order status is still Pending.
Display:
● Customer Name
● Order ID
● Order Amount
● Status

In [159]:
%sql select c.CustomerName,o.OrderID, o.OrderAmount, o.Status from Orders o join Customers c on o.CustomerID = c.CustomerID where o.OrderAmount>10000 and o.Status='Pending';


Running query in 'sqlite:///joins_assignment.db'

CustomerName,OrderID,OrderAmount,Status
Rohan Singh,5015,15500,Pending


**Question 4**
The customer success team wants to identify customers who joined more
than 365 days ago.
Display the number of days each customer has been associated with the
company.

In [162]:
%sql select CustomerID, CustomerName, JoinDate, cast(julianday('now') - julianday(JoinDate) as integer) as days_associated from Customers as c where (julianday('now') - julianday(JoinDate)) > 365;


Running query in 'sqlite:///joins_assignment.db'

CustomerID,CustomerName,JoinDate,days_associated
101,Rahul Sharma,2023-01-15,1381
102,Priya Mehta,2024-03-20,951
103,Arjun Nair,2025-01-12,653
104,Sneha Gupta,2022-07-10,1570
105,Kavya Reddy,2024-08-15,803
106,Amit Verma,2023-11-05,1087
107,Rohan Singh,2024-05-18,892
108,Simran Kaur,2025-02-01,633
109,Anjali Jain,2023-09-22,1131
110,Vikram Patel,2022-12-30,1397


Question 5
The finance department wants to estimate future revenue.
Create a report showing:
● Order ID
● Current Amount
● 18% GST
● Total Amount After GST
● Expected Collection Date (15 days after order date)

In [164]:
%%sql
SELECT
    OrderID as `Order_ID`,
    OrderAmount as `Current Amount`,
    (OrderAmount * 0.18) as `18% GST`,
    (OrderAmount * 1.18) as `Total Amount After GST`,
    date(OrderDate, '+15 days') as `Expected Collection Date`
FROM Orders as o;

Running query in 'sqlite:///joins_assignment.db'

Order_ID,Current Amount,18% GST,Total Amount After GST,Expected Collection Date
5001,12000,2160.0,14160.0,2025-01-20
5002,8500,1530.0,10030.0,2025-01-30
5003,22000,3960.0,25960.0,2025-02-25
5004,4500,810.0,5310.0,2025-03-02
5005,17500,3150.0,20650.0,2025-03-16
5006,9800,1764.0,11564.0,2025-03-25
5007,25000,4500.0,29500.0,2025-04-02
5008,14500,2610.0,17110.0,2025-04-17
5009,30000,5400.0,35400.0,2025-04-25
5010,6500,1170.0,7670.0,2025-04-30


**Question 6**
The operations team wants to review the customers ranked between 11th and
20th highest orders.
Use LIMIT and OFFSET.


In [166]:
 %sql select CustomerID,OrderID,OrderAmount from Orders as o order by OrderAmount desc limit 10 offset 10;


Running query in 'sqlite:///joins_assignment.db'

CustomerID,OrderID,OrderAmount
105,5006,9800
102,5002,8500
106,5013,7000
109,5010,6500
104,5004,4500


**Question 7**
An e-commerce company wants to identify customers who have never
placed any order despite registering on the platform.
Display:
● Customer ID
● Customer Name
● Join Date
● Number of Days Since Registration

In [168]:
%%sql
SELECT
    c.CustomerID,
    c.CustomerName,
    c.JoinDate,
    CAST(julianday('now') - julianday(c.JoinDate) AS INTEGER) AS DaysSinceRegistration
FROM Customers AS c
LEFT JOIN Orders AS o ON c.CustomerID = o.CustomerID
WHERE o.OrderID IS NULL;

Running query in 'sqlite:///joins_assignment.db'

CustomerID,CustomerName,JoinDate,DaysSinceRegistration
108,Simran Kaur,2025-02-01,633


**Question 8**
The HR department wants to analyze organizational hierarchy.
Using a JOIN, display:
● Employee Name
● Manager Name
● Department

In [169]:
%%sql select e.EmployeeID as "Employee ID",e.EmployeeName as "Employee Name",m.EmployeeName as ManagerName,e.Department as Department from Employees as e
left join Employees as m on e.ManagerID = m.EmployeeID;

Running query in 'sqlite:///joins_assignment.db'

Employee ID,Employee Name,ManagerName,Department
1,Rajesh Kumar,None,Management
2,Neha Sharma,Rajesh Kumar,Sales
3,Amit Gupta,Rajesh Kumar,IT
4,Priyanka Singh,Neha Sharma,Sales
5,Vikas Patel,Neha Sharma,Sales
6,Rohit Jain,Amit Gupta,IT
7,Anjali Verma,Amit Gupta,IT
8,Karan Mehta,Rajesh Kumar,HR
9,Sneha Kapoor,Karan Mehta,HR
10,Arjun Malhotra,Amit Gupta,IT


**Question 9**
The sales director wants a report of customers who placed orders within 30
days of joining the platform.
Display:
● Customer Name
● Join Date
● Order Date
● Days Taken To Place First Order

In [171]:
%%sql
SELECT
    c.CustomerName,
    c.JoinDate,
    MIN(o.OrderDate) AS FirstOrderDate,
    CAST(julianday(MIN(o.OrderDate)) - julianday(c.JoinDate) AS INTEGER) AS DaysTakenToPlaceFirstOrder
FROM Customers AS c
JOIN Orders AS o ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.JoinDate
HAVING (julianday(MIN(o.OrderDate)) - julianday(c.JoinDate)) < 30;


Running query in 'sqlite:///joins_assignment.db'

CustomerName,JoinDate,FirstOrderDate,DaysTakenToPlaceFirstOrder


**Question 9** The sales director wants a report of customers who placed orders more than 30 days of joining the platform. Display: ● Customer Name ● Join Date ● Order Date ● Days Taken To Place First Order

In [172]:
%%sql
SELECT
    c.CustomerName,
    c.JoinDate,
    MIN(o.OrderDate) AS FirstOrderDate,
    CAST(julianday(MIN(o.OrderDate)) - julianday(c.JoinDate) AS INTEGER) AS DaysTakenToPlaceFirstOrder
FROM Customers AS c
JOIN Orders AS o ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.JoinDate
HAVING (julianday(MIN(o.OrderDate)) - julianday(c.JoinDate)) > 30;


Running query in 'sqlite:///joins_assignment.db'

CustomerName,JoinDate,FirstOrderDate,DaysTakenToPlaceFirstOrder
Rahul Sharma,2023-01-15,2025-01-05,721
Priya Mehta,2024-03-20,2025-01-15,301
Arjun Nair,2025-01-12,2025-03-01,48
Sneha Gupta,2022-07-10,2025-02-15,951
Kavya Reddy,2024-08-15,2025-03-10,207
Amit Verma,2023-11-05,2025-03-18,499
Rohan Singh,2024-05-18,2025-04-02,319
Anjali Jain,2023-09-22,2025-04-15,571
Vikram Patel,2022-12-30,2025-05-01,853


**Question 10** Business Requirement Generate a report showing: ● Customer Name ● Total Purchase Amount ● Spending Category ● Loyalty Status Rules Spending Category ● Premium → Total Purchases > ₹50,000 ● Gold → ₹20,000 to ₹50,000 ● Silver → Less than ₹20,000 Loyalty Status ● Joined before 2024 → Loyal Customer ● Otherwise → New Customer

In [173]:
%%sql select c.CustomerName, sum(o.OrderAmount) as Total_Purchase_Amount, case when sum(o.OrderAmount) > 50000 then 'Premium'
when sum(o.OrderAmount) between 20000 and 50000 then 'Gold' else 'Silver' end as "Spending Category", case
when c.JoinDate < '2024-01-01' then 'Loyal Customer' else 'New Customer' end as "Loyalty Status" from Customers as c
join Orders as o on c.CustomerID = o.CustomerID group by c.CustomerID, c.CustomerName,c.JoinDate;

Running query in 'sqlite:///joins_assignment.db'

CustomerName,Total_Purchase_Amount,Spending Category,Loyalty Status
Rahul Sharma,64000,Premium,Loyal Customer
Priya Mehta,35500,Gold,New Customer
Arjun Nair,17500,Silver,New Customer
Sneha Gupta,22500,Gold,Loyal Customer
Kavya Reddy,9800,Silver,New Customer
Amit Verma,32000,Gold,Loyal Customer
Rohan Singh,30000,Gold,New Customer
Anjali Jain,6500,Silver,Loyal Customer
Vikram Patel,42000,Gold,Loyal Customer


**Question 11** : An online marketplace wants to find customers whose order
amount is higher than the average order amount of all orders.
Display:
● Customer Name
● Order ID
● Order Amount

In [174]:
%sql select c.CustomerName,o.OrderID,o.OrderAmount from Orders as o join Customers as c on o.CustomerID = c.CustomerID where o.OrderAmount>(select avg(o.OrderAmount) as average_OrderAmount from Orders as o);


Running query in 'sqlite:///joins_assignment.db'

CustomerName,OrderID,OrderAmount
Rahul Sharma,5003,22000
Arjun Nair,5005,17500
Amit Verma,5007,25000
Rahul Sharma,5009,30000
Vikram Patel,5011,42000
Sneha Gupta,5012,18000
Priya Mehta,5014,27000


**Question 12**
The management team wants the 2nd to 6th highest orders placed by
customers.
Display:
● Customer Name
● Order ID
● Order Amount

In [175]:
%%sql select c.CustomerName, o.OrderID, o.OrderAmount from orders as o join Customers as c on o.CustomerID = c.CustomerID
order by OrderAmount desc limit 5 offset 1;

Running query in 'sqlite:///joins_assignment.db'

CustomerName,OrderID,OrderAmount
Rahul Sharma,5009,30000
Priya Mehta,5014,27000
Amit Verma,5007,25000
Rahul Sharma,5003,22000
Sneha Gupta,5012,18000
